# KLUE NER SOTA Training - Stable Implementation

**Goal**: Beat KLUE paper's KoELECTRA baseline (F1 = 0.8611) using KoELECTRA-base-v3

## Techniques (Carefully Implemented for Stability)
1. **Multi-Sample Dropout** - 5 samples, averaged logits
2. **FGM Adversarial Training** - Embedding perturbation with safeguards
3. **R-Drop** - KL consistency with numerical stability
4. **Optimized Hyperparameters** - Based on KLUE paper



In [ ]:
# Cell 1: Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from torchcrf import CRF
import numpy as np
from tqdm.auto import tqdm
from seqeval.metrics import f1_score, classification_report
from seqeval.scheme import IOB2
import os
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2: Data Loading
dataset = load_dataset('klue', 'ner')
tokenizer = AutoTokenizer.from_pretrained('monologg/koelectra-base-v3-discriminator')

label_list = dataset['train'].features['ner_tags'].feature.names
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for i, l in enumerate(label_list)}
NUM_LABELS = len(label_list)
O_TAG_ID = label_to_id['O']

print(f'Labels ({NUM_LABELS}): {label_list}')

MAX_LENGTH = 128

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples['tokens'], truncation=True, is_split_into_words=True,
        max_length=MAX_LENGTH, padding='max_length'
    )
    labels = []
    for idx, tags in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=idx)
        prev_word = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word:
                label_ids.append(tags[word_idx])
            else:
                orig = tags[word_idx]
                name = id_to_label[orig]
                if name.startswith('B-'):
                    i_name = name.replace('B-', 'I-')
                    label_ids.append(label_to_id.get(i_name, orig))
                else:
                    label_ids.append(orig)
            prev_word = word_idx
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

tokenized = dataset.map(tokenize_and_align, batched=True)
tokenized = tokenized.remove_columns(dataset['train'].column_names)
tokenized.set_format('torch')
print(f'Train: {len(tokenized["train"])} | Val: {len(tokenized["validation"])}')

In [ ]:
# Cell 3: Model with Multi-Sample Dropout

class KoElectraNER(nn.Module):
    def __init__(self, model_name, num_labels, num_samples=5):
        super().__init__()
        self.num_labels = num_labels
        self.num_samples = num_samples
        
        self.electra = AutoModel.from_pretrained(model_name)
        hidden = self.electra.config.hidden_size
        
        self.lstm = nn.LSTM(hidden, 256, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(512, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
    
    def forward(self, input_ids, attention_mask, labels=None):
        enc = self.electra(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = enc.last_hidden_state
        lstm_out, _ = self.lstm(seq_out)
        
        if self.training and self.num_samples > 1:
            emissions_list = []
            for _ in range(self.num_samples):
                dropped = self.dropout(lstm_out)
                emissions_list.append(self.classifier(dropped))
            emissions = torch.stack(emissions_list, dim=0).mean(dim=0)
        else:
            emissions = self.classifier(self.dropout(lstm_out))
        
        mask = attention_mask.bool()
        
        if labels is not None:
            labels_crf = labels.clone()
            labels_crf[labels_crf == -100] = O_TAG_ID
            loss = -self.crf(emissions, labels_crf, mask=mask, reduction='mean')
            return loss, emissions
        return self.crf.decode(emissions, mask=mask)

print('Model class defined')

In [ ]:
# Cell 4: FGM Adversarial Training

class FGM:
    def __init__(self, model, eps=0.5):
        self.model = model
        self.eps = eps
        self.backup = {}
    
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and 'word_embeddings' in name:
                self.backup[name] = param.data.clone()
                if param.grad is not None:
                    norm = torch.norm(param.grad)
                    if norm > 0 and not torch.isnan(norm):
                        r = self.eps * param.grad / (norm + 1e-8)
                        param.data = param.data + r
    
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

print('FGM class defined')

In [ ]:
# Cell 5: R-Drop KL Loss

def rdrop_kl_loss(p_logits, q_logits, mask, alpha=1.0):
    p = F.log_softmax(p_logits, dim=-1)
    q = F.log_softmax(q_logits, dim=-1)
    p_prob = F.softmax(p_logits, dim=-1)
    q_prob = F.softmax(q_logits, dim=-1)
    
    kl_pq = F.kl_div(q, p_prob, reduction='none').sum(-1)
    kl_qp = F.kl_div(p, q_prob, reduction='none').sum(-1)
    
    mask_f = mask.float()
    kl = ((kl_pq + kl_qp) * mask_f).sum() / (mask_f.sum() + 1e-8)
    return alpha * kl

print('R-Drop loss defined')

In [ ]:
# Cell 6: Config

ENCODER_LR = 3e-5
HEAD_LR = 1e-3
EPOCHS = 20
BATCH_SIZE = 32
WARMUP = 0.1
GRAD_CLIP = 1.0
PATIENCE = 5

USE_FGM = True
FGM_EPS = 0.5
USE_RDROP = True
RDROP_ALPHA = 0.3

SAVE_DIR = './weights'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Config: LR={ENCODER_LR}, FGM={USE_FGM}, R-Drop={USE_RDROP}')

In [ ]:
# Cell 7: Initialize

model = KoElectraNER('monologg/koelectra-base-v3-discriminator', NUM_LABELS, num_samples=5)
model.to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)

optimizer = AdamW([
    {'params': model.electra.parameters(), 'lr': ENCODER_LR},
    {'params': model.lstm.parameters(), 'lr': HEAD_LR},
    {'params': model.classifier.parameters(), 'lr': HEAD_LR},
    {'params': model.crf.parameters(), 'lr': HEAD_LR},
])

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

if USE_FGM:
    fgm = FGM(model, eps=FGM_EPS)

print(f'Steps: {total_steps} | Warmup: {warmup_steps}')

In [ ]:
# Cell 7b: Resume from Checkpoint (OPTIONAL)
# Set RESUME_FROM_EPOCH to resume, or None to start fresh

RESUME_FROM_EPOCH = None  # e.g., 17

if RESUME_FROM_EPOCH:
    checkpoint_path = f'{SAVE_DIR}/checkpoint_epoch_{RESUME_FROM_EPOCH}.pt'
    if os.path.exists(checkpoint_path):
        print(f'Loading checkpoint from epoch {RESUME_FROM_EPOCH}...')
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Handle both old format (direct state_dict) and new format (dict with keys)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # New format
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            START_EPOCH = checkpoint['epoch'] + 1
            best_f1 = checkpoint.get('best_f1', 0.0)
        else:
            # Old format: checkpoint is the model state_dict directly
            model.load_state_dict(checkpoint)
            START_EPOCH = RESUME_FROM_EPOCH
            best_f1 = 0.0
            # Advance scheduler to correct position
            for _ in range(START_EPOCH * len(train_loader)):
                scheduler.step()
            print('  (Old checkpoint format - optimizer/scheduler reset)')
        
        print(f'Resuming from epoch {START_EPOCH}, best F1: {best_f1:.4f}')
    else:
        print(f'Checkpoint not found: {checkpoint_path}')
        START_EPOCH = 0
        best_f1 = 0.0
else:
    START_EPOCH = 0
    best_f1 = 0.0
    print('Starting from scratch')

In [ ]:
# Cell 8: Training Loop

patience_counter = 0

for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
    
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        loss1, emis1 = model(input_ids, attention_mask, labels)
        
        if USE_RDROP:
            loss2, emis2 = model(input_ids, attention_mask, labels)
            kl_loss = rdrop_kl_loss(emis1, emis2, attention_mask, RDROP_ALPHA)
            loss = 0.5 * (loss1 + loss2) + kl_loss
        else:
            loss = loss1
        
        if torch.isnan(loss):
            continue
        
        loss.backward()
        
        if USE_FGM:
            fgm.attack()
            adv_loss, _ = model(input_ids, attention_mask, labels)
            if not torch.isnan(adv_loss):
                adv_loss.backward()
            fgm.restore()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_labels = batch['labels']
            preds = model(input_ids, attention_mask)
            
            for p, l, m in zip(preds, batch_labels.numpy(), attention_mask.cpu().numpy()):
                length = m.sum()
                pred_tags = [label_list[t] for t, lab in zip(p[:length], l[:length]) if lab != -100]
                true_tags = [label_list[t] for t in l[:length] if t != -100]
                all_preds.append(pred_tags)
                all_labels.append(true_tags)
    
    f1 = f1_score(all_labels, all_preds, mode='strict', scheme=IOB2)
    print(f'Epoch {epoch+1}: Loss={avg_loss:.4f}, F1={f1:.4f}')
    
    # Save checkpoint (new format)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_f1': best_f1,
        'current_f1': f1
    }, f'{SAVE_DIR}/checkpoint_epoch_{epoch+1}.pt')
    
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pt')
        print(f'  -> New best!')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping')
            break
    
    torch.cuda.empty_cache()

print(f'\nBest F1: {best_f1:.4f}')

In [ ]:
# Cell 9: Final Evaluation

model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt'))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        batch_labels = batch['labels']
        preds = model(input_ids, attention_mask)
        
        for p, l, m in zip(preds, batch_labels.numpy(), attention_mask.cpu().numpy()):
            length = m.sum()
            pred_tags = [label_list[t] for t, lab in zip(p[:length], l[:length]) if lab != -100]
            true_tags = [label_list[t] for t in l[:length] if t != -100]
            all_preds.append(pred_tags)
            all_labels.append(true_tags)

print('='*60)
print('FINAL RESULTS')
print('='*60)
print(classification_report(all_labels, all_preds, mode='strict', scheme=IOB2, digits=4))
print(f'\nFinal F1: {f1_score(all_labels, all_preds, mode="strict", scheme=IOB2):.4f}')
print(f'Target: 0.8611')